# Selecting and filtering rows and columns

Python stage, lesson P12. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** pick columns by name and cells by label or position with `loc` and `iloc`; print a boolean mask, then filter rows with it; combine conditions with `&` and `|`; change values in one step with `.loc`
and see why a chained assignment changes nothing in pandas 3.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## A table with order ids as its index

In [ ]:
# Run this cell as it is. It makes orders_by_id: the orders table with order_id as its index, so that labels and positions differ.
orders_by_id = orders.set_index("order_id")
print(f"orders_by_id: {orders_by_id.shape[0]} rows, index labels {orders_by_id.index[0]} to {orders_by_id.index[-1]}.")

## Unit 1: Pick by name, by label and by position

**Predict first.** Order 1107 is the 107th row of `orders_by_id`, so its position is 106. Which of the last three prints show the time of order 1107, and which one does not?

In [ ]:
print(orders[["order_id", "ordered_at"]].head(3))
print(orders_by_id.loc[1107, "ordered_at"])
print(orders_by_id.iloc[106, 2])
print(orders_by_id.iloc[1107, 2])

## Unit 2: A mask, then a filter

**Predict first.** The first three orders were taken in stores 2, 1 and 2. What does the mask `orders["store_id"] == 2` hold for them? The shape of the filtered table is for you to read.

In [ ]:
campus = orders["store_id"] == 2
print(campus.head(3))
print(campus.shape, campus.sum())
campus_orders = orders[campus]
print(orders.shape, campus_orders.shape)

## Unit 3: Several conditions

**Predict first.** For the first three orders, which are in store 1 and paid by mobile? Which are in store 1 or paid by mobile? All three were paid by mobile, in stores 2, 1 and 2. The counts are for you to read.

In [ ]:
both = (orders["store_id"] == 1) & (orders["payment_method"] == "mobile")
either = (orders["store_id"] == 1) | (orders["payment_method"] == "mobile")
print(both.head(3).tolist())
print(either.head(3).tolist())
print(both.sum(), either.sum())

**Predict first.** Customer 15 placed 16 of the 1,800 orders, and one order has no customer. How many orders does each of the two counts give?

In [ ]:
not_15 = orders["customer_id"] != 15
print(not_15.sum())
print((not_15 & orders["customer_id"].notna()).sum())

## Unit 4: Change values in one step

**Predict first.** Harbor's mobile payments are relabeled `card`. Will the number of mobile orders in the table go up, go down or stay the same? The first number is the count before the change.

In [ ]:
work = orders.copy()
print((work["payment_method"] == "mobile").sum())
harbor_mobile = (work["store_id"] == 3) & (work["payment_method"] == "mobile")
work.loc[harbor_mobile, "payment_method"] = "card"
print((work["payment_method"] == "mobile").sum(), harbor_mobile.sum())

**Predict first.** The same relabeling on a fresh copy, written with two pairs of brackets. What does the count of mobile orders show afterwards?

In [ ]:
fresh = orders.copy()
mask = (fresh["store_id"] == 3) & (fresh["payment_method"] == "mobile")
fresh[mask]["payment_method"] = "card"
print((fresh["payment_method"] == "mobile").sum())

## Unit 5: Check: read the cells with a mask

**Predict first.** Which order_id has the status `canceled`?

In [ ]:
print(orders.loc[orders["status"] == "canceled", "order_id"].tolist())

## Finish it

### Exercise 1: Finish it

**Task:** How many orders did Harbor (store 3) take that were paid in cash? The mask for Harbor is written for you. Replace the first `None` with a mask for cash, then set `answer_1` to the number of orders that are both Harbor and cash.

**Expected:** one whole number.

In [ ]:
harbor = orders["store_id"] == 3
cash = None  # replace None: a mask that is True where payment_method is "cash"
answer_1 = None  # then set answer_1 to the number of orders that are both Harbor and cash

In [ ]:
check("Exercise 1", "answer_1", 55, traps={553: "553 counts the orders that are Harbor or cash. Which operator keeps only the orders that are both?"})

<details>
<summary>Hint</summary>

Unit 2 builds a mask with `==`, and unit 3 joins two masks. Count the `True` values with `sum()`.

</details>

<details>
<summary>Solution</summary>

```python
cash = orders["payment_method"] == "cash"
answer_1 = (harbor & cash).sum()
```

The answer is 55. With `|` the count is 553: every Harbor order plus every cash order.

</details>

## On your own

### Exercise 2: On your own

**Task:** What was the payment method of order 1500? Use `orders_by_id`, whose index holds the order ids. Before you start, say in one sentence what kind of value the answer is. Write the code in the empty cell below and set `answer_2`.

**Expected:** one word, a payment method.

In [ ]:
check("Exercise 2", "answer_2", "mobile", traps={"card": "That is the payment method of the 1,501st row, which is order 2501. Order 1500 is a label of the index, not a position."})

<details>
<summary>Hint</summary>

Unit 1 picks one cell by label with `.loc[row label, column name]`. In `orders_by_id` the row labels are order ids.

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = orders_by_id.loc[1500, "payment_method"]
```

The answer is `'mobile'`. `.iloc[1500, 4]` counts positions from 0, so it reads the 1,501st row, order 2501, and gives `'card'`.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** A colleague wants the payment method of the canceled order to read `none`. The line below gives no error, but nothing changes. Fix it, then set `answer_3` to the number of orders in `work` whose payment method is `none`.

**Expected:** one whole number.

In [ ]:
work = orders.copy()
work[work["status"] == "canceled"]["payment_method"] = "none"  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 to the number of orders in work whose payment_method is "none"

In [ ]:
check("Exercise 3", "answer_3", 1, traps={0: "Nothing changed: the assignment went to a temporary copy. Name the rows and the column in one step."})

<details>
<summary>Hint</summary>

Unit 4 changes values with one `.loc` that holds the mask and the column name. Two pairs of brackets change a temporary table.

</details>

<details>
<summary>Solution</summary>

```python
work.loc[work["status"] == "canceled", "payment_method"] = "none"
answer_3 = (work["payment_method"] == "none").sum()
```

The answer is 1: the one canceled order. The colleague's line changed a temporary copy of the canceled row, so the count stayed 0.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")